# LLM Eval - Testing PromptRegistry & LLMTracer

This notebook tests the `prompts.py` and `tracer.py` modules with both mock and real LLM calls.

In [82]:
import sys
sys.path.insert(0, '..')

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv("../.env")  # Load from project root

True

In [ ]:
from llm_eval.prompts import PromptRegistry
from llm_eval.tracer import LLMTracer
from llm_eval.store import TraceStore
from llm_eval.utils.models import ModelSelector

## 1. Test PromptRegistry

In [84]:
# Initialize registry (creates/uses test_registry.db)
registry = PromptRegistry("test_registry.db")
print("Registry initialized!")

Registry initialized!


In [85]:
# Register first version of a prompt
prompt_v1 = registry.register(
    name="summarizer",
    template="You are a helpful assistant. Summarize the following text: {text}",
    description="Initial summarizer prompt"
)

print(f"Registered: {prompt_v1.name} v{prompt_v1.version}")
print(f"Hash: {prompt_v1.template_hash[:16]}...")

Registered: summarizer v1
Hash: 29f02c129c9e8262...


In [86]:
# Register same template again - should return existing (dedup)
prompt_same = registry.register(
    name="summarizer",
    template="You are a helpful assistant. Summarize the following text: {text}",
    description="This description is ignored since template is same"
)

print(f"Same template returned: v{prompt_same.version} (should be v1)")
assert prompt_same.version == 1, "Should return existing version!"

Same template returned: v1 (should be v1)


In [87]:
# Register new version with different template
prompt_v2 = registry.register(
    name="summarizer",
    template="You are an expert summarizer. Create a concise, accurate summary of: {text}",
    description="Improved prompt with expert framing"
)

print(f"New version: {prompt_v2.name} v{prompt_v2.version}")
assert prompt_v2.version == 2, "Should be version 2!"

New version: summarizer v2


In [88]:
# List all versions
versions = registry.list_versions("summarizer")
print(f"Found {len(versions)} versions:")
for v in versions:
    print(f"  v{v.version}: {v.description}")

Found 2 versions:
  v2: Improved prompt with expert framing
  v1: Initial summarizer prompt


In [89]:
# Get specific version
v1 = registry.get("summarizer", version=1)
print(f"Got v1: {v1.template[:50]}...")

# Get latest (default)
latest = registry.get("summarizer")
print(f"Latest is v{latest.version}")

Got v1: You are a helpful assistant. Summarize the followi...
Latest is v2


In [90]:
# Test KeyError for non-existent prompt
try:
    registry.get("nonexistent")
except KeyError as e:
    print(f"Correctly raised KeyError: {e}")

Correctly raised KeyError: "Prompt 'nonexistent' not found"


## 2. Test LLMTracer (without actual LLM call)

In [91]:
# Initialize tracer
tracer = LLMTracer(
    db_path="test_registry.db",  # Same DB as registry
    project="test_project",
    session_id="test_session_001"
)

print(f"Tracer initialized!")
print(f"  Project: {tracer.project}")
print(f"  Session: {tracer.session_id}")

Tracer initialized!
  Project: test_project
  Session: test_session_001


In [92]:
# Test new_session
new_sess = tracer.new_session()
print(f"New session (auto UUID): {new_sess}")

named_sess = tracer.new_session("my_experiment_001")
print(f"Named session: {named_sess}")

New session (auto UUID): 4813951d-7d9d-4737-a1dc-81c6f263b5a3
Named session: my_experiment_001


In [93]:
# Test set_prompt_context
tracer.set_prompt_context("summarizer", version=1)
print(f"Prompt context set to: {tracer._prompt_name} v{tracer._prompt_version}")

Prompt context set to: summarizer v1


In [94]:
# Test error when prompt not set
tracer2 = LLMTracer(db_path="test_registry.db", project="test")
from uuid import uuid4

try:
    tracer2.on_llm_start(
        serialized={},
        prompts=["test"],
        run_id=uuid4()
    )
except RuntimeError as e:
    print(f"Correctly raised RuntimeError: {e}")

Correctly raised RuntimeError: set_prompt_context() must be called before LLM invocation. No prompt context has been set.


## 3. Test LLMTracer with Mock LLM Call

In [95]:
from uuid import uuid4
from unittest.mock import MagicMock
import time

# Create a fresh tracer
tracer = LLMTracer(
    db_path="test_registry.db",
    project="mock_test",
    session_id="mock_session_001"
)
tracer.set_prompt_context("summarizer", version=2)

# Simulate on_llm_start
run_id = uuid4()
tracer.on_llm_start(
    serialized={"name": "ChatOpenAI"},
    prompts=[],
    run_id=run_id,
    messages=[
        {"role": "system", "content": "You are an expert summarizer."},
        {"role": "user", "content": "Summarize: The quick brown fox jumps over the lazy dog."}
    ]
)
print(f"on_llm_start called with run_id: {run_id}")
print(f"Pending calls: {len(tracer._pending)}")

on_llm_start called with run_id: f0f3917b-0d6f-407f-b97e-9d51f23cdea5
Pending calls: 1


In [96]:
# Simulate some processing time
time.sleep(0.1)

# Create mock LLMResult
mock_generation = MagicMock()
mock_generation.text = "A fox jumps over a dog."
mock_generation.message = MagicMock()
mock_generation.message.content = "A fox jumps over a dog."
mock_generation.message.usage_metadata = {
    "input_tokens": 25,
    "output_tokens": 8
}
mock_generation.message.response_metadata = {
    "model_name": "gpt-4"
}

mock_response = MagicMock()
mock_response.generations = [[mock_generation]]
mock_response.llm_output = {
    "token_usage": {
        "prompt_tokens": 25,
        "completion_tokens": 8,
        "total_tokens": 33
    },
    "model_name": "gpt-4"
}

# Call on_llm_end
tracer.on_llm_end(response=mock_response, run_id=run_id)
print("on_llm_end called")
print(f"Pending calls after end: {len(tracer._pending)}")

on_llm_end called
Pending calls after end: 0


In [97]:
# Verify trace was saved
store = TraceStore("test_registry.db")
traces = store.get_traces({"session_id": "mock_session_001"})

print(f"Found {len(traces)} trace(s):")
for t in traces:
    print(f"  ID: {t['id'][:8]}...")
    print(f"  Prompt: {t['prompt_name']} v{t['prompt_version']}")
    print(f"  Output: {t['output_content']}")
    print(f"  Tokens: {t['total_tokens']}")
    print(f"  Latency: {t['latency_ms']}ms")
    print(f"  Status: {t['status']}")

Found 1 trace(s):
  ID: 7feb7895...
  Prompt: summarizer v2
  Output: A fox jumps over a dog.
  Tokens: 33
  Latency: 109ms
  Status: success


## 4. Test Error Handling

In [98]:
# Test on_llm_error
tracer.new_session("error_test_session")
tracer.set_prompt_context("summarizer")

error_run_id = uuid4()
tracer.on_llm_start(
    serialized={},
    prompts=["Test prompt"],
    run_id=error_run_id
)

# Simulate error
time.sleep(0.05)
tracer.on_llm_error(
    error=Exception("API rate limit exceeded"),
    run_id=error_run_id
)

print("Error trace saved")

Error trace saved


In [99]:
# Verify error trace
error_traces = store.get_traces({"session_id": "error_test_session"})
print(f"Error traces: {len(error_traces)}")
for t in error_traces:
    print(f"  Status: {t['status']}")
    print(f"  Error: {t['error']}")

Error traces: 1
  Status: error
  Error: API rate limit exceeded


---

## 5. Real LLM Call with ModelSelector

This section tests the tracer with actual LLM calls using the ModelSelector which provides access to multiple providers (LLM Gateway, Azure OpenAI, Bedrock).

In [100]:
# Register a prompt for real LLM testing
real_prompt = registry.register(
    name="qa_assistant",
    template="""You are a helpful assistant. Answer the user's question concisely.

Question: {question}

Answer:""",
    description="Simple QA prompt for testing"
)
print(f"Registered: {real_prompt.name} v{real_prompt.version}")

Registered: qa_assistant v1


In [ ]:
# Show available models
print("Available Gateway model aliases:")
for alias in list(ModelSelector.GATEWAY_MODEL_ALIASES.keys())[:10]:
    print(f"  {alias}")
print("  ...")

In [ ]:
# Create tracer for real LLM calls
real_tracer = LLMTracer(
    db_path="test_registry.db",
    project="real_llm_test",
    session_id="real_session_001"
)
real_tracer.set_prompt_context("qa_assistant")

print(f"Tracer ready for real LLM calls")
print(f"  Project: {real_tracer.project}")
print(f"  Session: {real_tracer.session_id}")

In [ ]:
# Get model with tracer callback using ModelSelector
llm = ModelSelector.get_model("gpt-4o-mini", callbacks=[real_tracer])
print(f"Model: gpt-4o-mini with tracer attached")

In [ ]:
# Make a real LLM call
messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
    {"role": "user", "content": "What is the capital of France?"}
]

response = llm.invoke(messages)
print(f"Response: {response.content}")

In [ ]:
# Make a real LLM call
messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
    {"role": "user", "content": "What is the capital of France?"}
]

response = llm.invoke(messages)
print(f"Response: {response.content}")

In [ ]:
# Start a new session and try a different model (Claude)
real_tracer.new_session("claude_test_session")
real_tracer.set_prompt_context("qa_assistant")

# Get Claude model using ModelSelector
claude_llm = ModelSelector.get_model("claude-3.5-haiku", callbacks=[real_tracer])

messages3 = [
    {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
    {"role": "user", "content": "Explain photosynthesis in one sentence."}
]

response3 = claude_llm.invoke(messages3)
print(f"Claude response: {response3.content}")

In [ ]:
# Start a new session and try a different model
real_tracer.new_session("claude_test_session")
real_tracer.set_prompt_context("qa_assistant")

# Get Claude model
claude_llm = gateway.get_model("claude-3.5-haiku", callbacks=[real_tracer])

messages3 = [
    {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
    {"role": "user", "content": "Explain photosynthesis in one sentence."}
]

response3 = claude_llm.invoke(messages3)
print(f"Claude response: {response3.content}")

In [ ]:
# Verify all real traces were saved
print("=== Traces from real_session_001 (GPT-4o-mini) ===")
gpt_traces = store.get_traces({"session_id": "real_session_001"})
for t in gpt_traces:
    print(f"\nTrace: {t['id'][:8]}...")
    print(f"  Model: {t['model_name']}")
    print(f"  Input tokens: {t['input_tokens']}")
    print(f"  Output tokens: {t['output_tokens']}")
    print(f"  Latency: {t['latency_ms']}ms")
    print(f"  Output: {t['output_content'][:100]}..." if len(t['output_content']) > 100 else f"  Output: {t['output_content']}")

In [ ]:
print("\n=== Traces from claude_test_session (Claude) ===")
claude_traces = store.get_traces({"session_id": "claude_test_session"})
for t in claude_traces:
    print(f"\nTrace: {t['id'][:8]}...")
    print(f"  Model: {t['model_name']}")
    print(f"  Input tokens: {t['input_tokens']}")
    print(f"  Output tokens: {t['output_tokens']}")
    print(f"  Latency: {t['latency_ms']}ms")
    print(f"  Output: {t['output_content']}")

## 6. View All Sessions Summary

In [ ]:
sessions = store.get_sessions()
print(f"Total sessions: {len(sessions)}\n")
print(f"{'Session':<30} {'Traces':>7} {'Success':>8} {'Errors':>7} {'Tokens':>10}")
print("-" * 65)
for s in sessions:
    session_name = s['session_id'][:28] + ".." if len(s['session_id']) > 30 else s['session_id']
    print(f"{session_name:<30} {s['trace_count']:>7} {s['success_count']:>8} {s['error_count']:>7} {s['total_tokens']:>10}")

## Cleanup

In [ ]:
# Uncomment to delete test database
# import os
# os.remove("test_registry.db")
# print("Deleted test_registry.db")